# Gauss-Newton Matrix

In [9]:
import numpy as np
from scipy.spatial.transform import Rotation as R

## COE2RV

NOTE: Need to use semiparameter ($p$) instead of semimajor axis ($a$). $a$ is infinite for the parabola, whereas $p$ is defined for all orbits

In [ ]:
def COE2RV(coe, mu=3.986004418*(10**14)):

    # INPUT: 
    #   coe is an array of the Keplerian Orbital Elements
    #       a - semi-major axis
    #       e - eccentricity
    #       i - inclination
    #       node - right ascension of the ascending node
    #       arg - argument of perigee
    #       nu - true anomaly
    #   mu - gravitational parameters (=GM). Default set to the value for Earth

    # OUTPUT:
    #   r - position vector of satellite
    #   v - velocity vector of satellite


    a, e, i, node, arg, nu = coe

    sin_nu = np.sin(np.deg2rad(nu))
    cos_nu = np.cos(np.deg2rad(nu))

    # Calculate semiparameter (p)
    p = a * (1-e**2)

    # Perifocal Coordinate System
    R_PQW = np.zeros(3)
    V_PQW = np.zeros(3)

    R_PQW[0] = (p * cos_nu) / (1 + e*cos_nu)
    R_PQW[1] = (p * sin_nu) / (1 + e*cos_nu)
    
    V_PQW[0] = -1 * np.sqrt(mu/p) * sin_nu
    V_PQW[1] = np.sqrt(mu/p) * (e + cos_nu)

    # Rotation Matrix
    R_node_z = R.from_euler('z', np.deg2rad(node)).as_matrix()
    R_i_x    = R.from_euler('x', np.deg2rad(i)).as_matrix()
    R_arg_z  = R.from_euler('z', np.deg2rad(arg)).as_matrix()
    R_total  = R_node_z @ R_i_x @ R_arg_z

    # Rotate Perifocal to IJK
    R_IJK = R_total @ R_PQW
    V_IJK = R_total @ V_PQW

    return R_IJK, V_IJK

### Example 2.6 from FoAaA (pg. 119)

Verifying function is correct

In [15]:
p = 11067.790 #km
e = 0.83285
i = 87.87
node = 227.89
arg = 53.38
nu = 92.335

a = (p*1000) / (1-e**2)

coe = [a, e, i, node, arg, nu]

r, v = COE2RV(coe)

print("Vector r (m):")
print("Textbook: [6525344  6861535  6449125]")
print("Mine:    ", r)

print("\nVector v (m/s):")
print("Textbook: [4902.276  5533.124  -1975.709]")
print("Mine:    ", v)

Vector r (m):
Textbook: [6525344  6861535  6449125]
Mine:     [6525368.12098609 6861531.83489605 6449118.61416016]

Vector v (m/s):
Textbook: [4902.276  5533.124  -1975.709]
Mine:     [ 4902.27864642  5533.13956836 -1975.71009954]


## Calculate $\nu$ from $M_0$

- $n$ is the mean motion
$$
M(t) = M_0 + n (t-t_0)
$$
$$
n = \sqrt{\frac{\mu}{a^3}}
$$
- Use Newton-Raphson Method (Algorithm 2 in FoAaA pg. 65) to find $E$
	- Converts $M=E-e\sin (E)$ to be in terms of $E$
    - Tolerance set to $10^{-8}$ as according to textbook
- The calculate $\nu$
$$
\cos(\nu) = \frac{\cos(E) - e}{1-e\cos(E)}
$$

In [ ]:
def n(a, mu=3.986004418*(10**14)):
    # Returns the mean motion in rads/s
    return np.sqrt(mu / a**3)

In [40]:
def NewtRaph(M, e, tolerance=10**-8):
    # Converts single pair of M and e to E
    # Inputs:
    #   M - Mean anomaly (degrees)
    #   e - Eccentricity
    #   tolerance - default to 10^8
    # Outputs:
    #   E - Eccentric anomaly
    
    M_rad = np.deg2rad(M)

    if (M_rad>-np.pi and M_rad<0) or (M_rad>np.pi):
        E = M_rad-e
    else:
        E = M_rad+e

    while True:
        sin_E = np.sin(E)
        cos_E = np.cos(E)
        nextE = E + (M_rad-E+e*sin_E)/(1-e*cos_E)

        abs_diff = abs(nextE-E)
        E = nextE

        if abs_diff < tolerance:
            break

    return E

In [ ]:
def MultiNewtRaph(t, M_0, n, e, tolerance=10**-8):
    # Inputs:
    #   t - array of time entries (s)
    #   M_0 - initial mean anomaly (deg)
    #   n - mean motion (rads/s) 
    #   e - eccentricity
    # Outputs:
    #   nu_t - true anomaly (in radians) at each point in time

    t_0 = t[0]

    # Mean anomalies
    M_t = np.zeros_like(t)
    M_t = M_0 + n*(t-t_0)

    # Newton-Raphson to find eccentric anomaly (E)
    E_t = np.zeros_like(M_t)
    for i in range(len(M_t)):
        E_t[i] = NewtRaph(M[i], e, tolerance=tolerance)

    return E_t

In [ ]:
def nu(E, e):
    # Inputs:
    #   E - Eccentric anomaly (radians)
    #   e - Eccentricity
    # Outputs:
    #   nu - True Anomaly (radians)
     
    cosE = np.cos(E)
    nu = np.arccos( (cosE - e)/(1 - e*cosE) ) # in radians
    return nu

### Newton-Raphson: Testing Example 2-1 from FoAaA

In [42]:
M = 235.4
e = 0.4
tolerance = 10**-8

E = NewtRaph(M, e)

print("Expect E (deg): 220.512074767522")
print("Final E (deg): ", np.rad2deg(E))

Expect E (deg): 220.512074767522
Final E (deg):  220.51207476752208


---
## Groundstation Vectors

<font color='red'>TODO</font>

Position $\textbf{r}_{gs}$
1. Convert geodetic coordinates to (lat/lon/att) to ECEF
2. Rotate ECEF to ECI using Earth's rotation angle at each timestamp (via sidereal time)

Velocity $\textbf{v}_{gs}$
1. Differentiate rotation (angular velocity crossed with position)

---
## Doppler Shift from $\textbf{r}$ and $\textbf{v}$

In [ ]:
def f_D(r, v, r_gs, v_gs, f_c, c=299792458):
    rho = r - r_gs
    rho_hat = rho / np.sqrt(rho.dot(rho))

    v_rel = v - v_gs

    k = f_c/c

    f_D = k * rho_hat * (-1 * v_rel)

    return f_D

# Find Doppler Shift Error

In [ ]:
def residual(y_pred, y_true):
    return y_pred-y_true